# HUẤN LUYỆN VÀ TINH CHỈNH (FINE-TUNING) MÔ HÌNH BARTPHO CHO TÓM TẮT TIN TỨC TIẾNG VIỆT

Notebook này thực hiện tinh chỉnh mô hình pre-trained **BARTpho** (`vinai/bartpho-word`) trên tập dữ liệu tin tức báo chí tiếng Việt theo lộ trình 5 bước cốt lõi.

## BƯỚC 1: CHUẨN BỊ MÔI TRƯỜNG VÀ DỮ LIỆU

* **Môi trường**: Đảm bảo bạn đang chạy trên GPU (Google Colab hoặc Kaggle).
* **Dữ liệu**: Nạp các tập dữ liệu Train, Validation và Test sạch đã được tiền xử lý.

In [ ]:
import os
import sys
import torch
import pandas as pd
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device}")
if device == "cuda":
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

Đang sử dụng thiết bị: cuda
Tên GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_path = "/content/drive/MyDrive/AI/processed/train_clean.xlsx"
val_path = "/content/drive/MyDrive/AI/processed/val_clean.xlsx"
test_path = "/content/drive/MyDrive/AI/processed/new_test_clean.xlsx"

if not os.path.exists(train_path):
    train_path = "data/processed/train_clean.xlsx"
    val_path = "data/processed/validation_clean.xlsx"
    test_path = "data/processed/new_test_clean.xlsx"

print("Đang tải các tập dữ liệu sạch...")
train_df = pd.read_excel(train_path)
val_df = pd.read_excel(val_path)
test_df = pd.read_excel(test_path)

print(f"[Dữ liệu huấn luyện (Train)]: {len(train_df):,} mẫu")
print(f"[Dữ liệu giám sát (Validation)]: {len(val_df):,} mẫu")
print(f"[Dữ liệu kiểm thử (Test)]: {len(test_df):,} mẫu")

Đang tải các tập dữ liệu sạch...
[Dữ liệu huấn luyện (Train)]: 14,129 mẫu
[Dữ liệu giám sát (Validation)]: 1,422 mẫu
[Dữ liệu kiểm thử (Test)]: 936 mẫu


## BƯỚC 2: KHỞI TẠO MÔ HÌNH VÀ BỘ MÃ HÓA (TOKENIZER)

Sử dụng mô hình **BARTpho** (`vinai/bartpho-word`) được phát triển bởi VinAI cho tiếng Việt, dựa trên kiến trúc gốc BART-large.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_checkpoint = "vinai/bartpho-word"
print(f"Đang nạp Tokenizer từ checkpoint: {model_checkpoint}...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print(f"Đang nạp Mô hình Seq2Seq từ checkpoint: {model_checkpoint}...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

model.generation_config.num_beams = 4
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.length_penalty = 1.0
model.to(device)
print("Nạp mô hình và bộ mã hóa thành công!")

Đang nạp Tokenizer từ checkpoint: vinai/bartpho-word...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

...

Đang nạp Mô hình Seq2Seq từ checkpoint: vinai/bartpho-word...


pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Nạp mô hình và bộ mã hóa thành công!


model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

## BƯỚC 3: THỰC HIỆN MÃ HÓA (TOKENIZATION)

Ánh xạ văn bản thô tiếng Việt thành chuỗi IDs tương ứng. Thiết lập độ dài đầu vào tối đa là **512** (giảm từ 1024 để tiết kiệm VRAM) và đầu ra tóm tắt tối đa là **256** (bao phủ 99% độ dài summary thực tế). Sử dụng **dynamic padding** thay vì `max_length` padding — DataCollator sẽ tự padding theo batch ngắn nhất, giảm tính toán thừa đáng kể.

In [ ]:
def get_token_lengths(texts):
    lengths = []
    for text in texts:
        tokens = tokenizer.encode(
            str(text),
            add_special_tokens=True
        )
        lengths.append(len(tokens))
    return lengths

print("Đang tính độ dài token...")

train_content_lengths = get_token_lengths(train_df["content"])
train_summary_lengths = get_token_lengths(train_df["summary"])

val_content_lengths = get_token_lengths(val_df["content"])
val_summary_lengths = get_token_lengths(val_df["summary"])

Đang tính độ dài token...


In [ ]:
import numpy as np

def show_stats(name, lengths):
    print(f"\n{name}")
    print(f"Min      : {np.min(lengths)}")
    print(f"Mean     : {np.mean(lengths):.2f}")
    print(f"Median   : {np.median(lengths):.2f}")
    print(f"95%      : {np.percentile(lengths, 95):.0f}")
    print(f"99%      : {np.percentile(lengths, 99):.0f}")
    print(f"Max      : {np.max(lengths)}")

show_stats("TRAIN CONTENT", train_content_lengths)
show_stats("TRAIN SUMMARY", train_summary_lengths)

show_stats("VALID CONTENT", val_content_lengths)
show_stats("VALID SUMMARY", val_summary_lengths)


TRAIN CONTENT
Min      : 31
Mean     : 504.93
Median   : 446.00
95%      : 1029
99%      : 1132
Max      : 1570

TRAIN SUMMARY
Min      : 12
Mean     : 131.31
Median   : 126.00
95%      : 216
99%      : 249
Max      : 344

VALID CONTENT
Min      : 41
Mean     : 498.12
Median   : 447.50
95%      : 1011
99%      : 1131
Max      : 1415

VALID SUMMARY
Min      : 21
Mean     : 129.94
Median   : 126.00
95%      : 206
99%      : 245
Max      : 292


In [ ]:
def preprocess_tokenize_function(examples):
    model_inputs = tokenizer(
        examples["content"],
        max_length=512,
        padding=False,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=256,
        padding=False,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print("Đang thực hiện mã hóa Tokenization trên toàn bộ các tập dữ liệu...")
tokenized_train = train_dataset.map(preprocess_tokenize_function, batched=True)
tokenized_val = val_dataset.map(preprocess_tokenize_function, batched=True)
tokenized_test = test_dataset.map(preprocess_tokenize_function, batched=True)
print("Mã hóa dữ liệu hoàn tất!")

Đang thực hiện mã hóa Tokenization trên toàn bộ các tập dữ liệu...


Map:   0%|          | 0/14129 [00:00<?, ? examples/s]

Map:   0%|          | 0/1422 [00:00<?, ? examples/s]

Map:   0%|          | 0/936 [00:00<?, ? examples/s]

Mã hóa dữ liệu hoàn tất!


## BƯỚC 4: THIẾT LẬP THAM SỐ VÀ KỸ THUẬT FINE-TUNING CHUYÊN SÂU

Cấu hình các siêu tham số huấn luyện tối ưu cho GPU hạn chế, áp dụng các kỹ thuật: **Dynamic Padding, Gradient Checkpointing, Adafactor Optimizer, Gradient Accumulation, Label Smoothing, Beam Search và Warmup**.

In [ ]:
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/AI/results_bartpho",

    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    weight_decay=0.01,
    num_train_epochs=3,
    warmup_ratio=0.1,
    label_smoothing_factor=0.1,
    predict_with_generate=True,
    generation_max_length=150,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),
    optim="adafactor",
    logging_steps=100,
    save_total_limit=4,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    report_to="none"
)
print("Cấu hình tham số huấn luyện nâng cao hoàn tất!")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Cấu hình tham số huấn luyện nâng cao hoàn tất!


In [ ]:
!pip install transformers datasets evaluate sentencepiece rouge-score openpyxl accelerate>=0.21.0 -q

import numpy as np
import evaluate

rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    gen_lengths = [len(pred.split()) for pred in decoded_preds]
    result["gen_len"] = round(float(np.mean(gen_lengths)), 2)
    return {
        k: round(v * 100, 4) if k != "gen_len" else v
        for k, v in result.items()
    }

## BƯỚC 5: HUẤN LUYỆN, GIÁM SÁT VÀ LƯU TRỮ

Khởi tạo `Seq2SeqTrainer`, tích hợp cơ chế **Early Stopping** và tiến hành huấn luyện.

In [ ]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.0
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

print("Bộ huấn luyện Seq2SeqTrainer đã sẵn sàng!")

Bộ huấn luyện Seq2SeqTrainer đã sẵn sàng!


In [ ]:
print("🚀 Bắt đầu quá trình huấn luyện mô hình BARTpho...")
train_result = trainer.train()
print("🎉 Huấn luyện thành công!")

🚀 Bắt đầu quá trình huấn luyện mô hình BARTpho...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
500,22.777532,2.622461,72.715400,49.942500,50.373200,50.397100,114.970000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]